# Named Entity Linking (NEL) avec SpaCy

Pré-annoter la VT NER afin de produire une VT NEL.

Ce notebook réalise de l'**entity linking** sur des occurrences de noms de personnes extraites de fichiers JSON,  
en s'appuyant sur votre **base de connaissances SQLite** (table `persons`) comme référentiel d'entités.

### Flux de traitement
```
Fichiers JSON (occurrences) ──► SpaCy NLP ──► Entity Linking ──► SQLite KB (table persons) ──► JSON enrichi
```

### Schéma SQLite attendu (table `persons`)
| colonne | description |
|---|---|
| `person_id` | identifiant unique (PK) |
| `last_name` | nom de famille |
| `first_name` | prénom |
| `alias` | alias / variantes orthographiques |
| `birth_date` / `death_date` | dates |
| `wikidata_qid` | identifiant Wikidata |
| `wikipedia_url` | URL Wikipédia |
| `sycomore_id` | identifiant base Sycomore (Assemblée nationale) |
| `senat_id` | identifiant base Sénat |

## Description de la méthode

**La cascade en trois niveaux**

Le moteur de linking fonctionne comme une cascade : on tente d'abord le niveau le plus fiable, et on ne descend au niveau suivant que si le précédent échoue. Pour chaque `str_ref`, on vérifie d'abord s'il correspond *exactement* (après mise en minuscules) à un nom ou alias enregistré dans la base — c'est le cas le plus fréquent pour des noms complets bien orthographiés comme "Paul Doumer". Si rien ne correspond exactement, on passe au matching flou ; si le flou laisse plusieurs candidats proches et ambigus, on fait appel aux vecteurs pour trancher. Cette architecture privilégie la précision : on ne mobilise les méthodes plus coûteuses et plus incertaines que lorsque c'est nécessaire.

**Le matching flou (RapidFuzz)**

Pour le flou, j'utilise le scorer `token_sort_ratio` de RapidFuzz plutôt qu'un simple ratio de Levenshtein. Son intérêt est qu'il découpe la chaîne en tokens (mots), les trie par ordre alphabétique, puis compare les deux chaînes ainsi normalisées. Cela rend le score insensible à l'ordre des mots — "Vaillant Édouard" et "Édouard Vaillant" obtiendraient un score quasi parfait, alors qu'un Levenshtein brut les pénaliserait fortement. `process.extract` cherche ensuite, parmi *tous* les noms et alias indexés (la propriété `all_names` de chaque `PersonEntity` : nom complet, nom seul, "P. Nom", alias déclarés), les `TOP_K_CANDIDATES` meilleurs scores au-dessus du seuil `FUZZY_THRESHOLD` (80/100 par défaut). Comme une même personne peut apparaître sous plusieurs alias proches, on déduplique ensuite les candidats par `person_id` en gardant le meilleur score par entité. Si un seul candidat survit, ou si le meilleur dépasse 0,95, on le retient directement — pas besoin d'aller plus loin.

**La désambiguïsation vectorielle**

Ce niveau n'intervient que dans le cas résiduel où plusieurs candidats restent proches en score flou (par exemple "Thierry" pourrait fuzzy-matcher plusieurs personnes de noms voisins). On calcule alors le vecteur SpaCy (les embeddings statiques du modèle `fr_core_news_lg`, de type word2vec/floret) de la chaîne recherchée, et celui du nom canonique de chaque candidat, puis on mesure leur similarité cosinus. Cette similarité est combinée au score flou (60 % flou / 40 % cosinus) pour produire un score final, et le candidat avec le meilleur score combiné est choisi. Il faut toutefois rester honnête sur les limites de cette étape : les vecteurs statiques de SpaCy sont entraînés sur du texte général et n'encodent pas une "sémantique des patronymes" très discriminante — deux noms de famille rares auront souvent des vecteurs proches simplement parce qu'ils sont tous deux hors-vocabulaire et traités via des sous-mots similaires. Ce score sert donc surtout de **tie-breaker déterministe** plutôt que d'un signal sémantique fort.

**Par rapport à l'entity linking "natif" de SpaCy**

Il est important de préciser que le pipeline principal (sections 8-9) n'utilise *pas* le composant entraîné `entity_linker` de SpaCy, même si on construit bien un objet `KnowledgeBase` au format SpaCy (section 7). Le composant natif `entity_linker` est un modèle supervisé qui apprend, à partir d'exemples annotés (mention + *phrase contenant la mention* → entity_id), à choisir la bonne entité en fonction du contexte de la phrase. Dans votre cas, les occurrences sont des `str_ref` isolées, sans phrase autour, et vous n'avez (a priori) pas de jeu d'exemples annotés — ce composant contextuel serait donc peu pertinent ou nécessiterait un travail d'annotation important. La cascade exact/flou/vecteur que j'ai implémentée est une alternative légère, sans entraînement, bien adaptée à ce cas d'usage. La section 12 (optionnelle) reste disponible si vous constituez un jour un corpus annoté et souhaitez exploiter le contexte phrastique pour affiner la désambiguïsation.

## 1. Installation des dépendances

In [ ]:
# Installation de SpaCy (dernière version) et du modèle français
!pip install -q --upgrade spacy
!python -m spacy download fr_core_news_lg

# Librairies utilitaires
!pip install -q rapidfuzz tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 MB 2.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.0 MB/s eta 0:00:00


In [ ]:
import spacy
print(f"SpaCy version : {spacy.__version__}")

SpaCy version : 3.8.14


## 2. Import des bibliothèques

In [ ]:
import json
import sqlite3
import os
import glob
import csv
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field

import numpy as np
import spacy
from spacy.kb import InMemoryLookupKB
from spacy.kb import KnowledgeBase, Candidate
from spacy.language import Language

from rapidfuzz import fuzz, process
from tqdm import tqdm

print("✅ Imports OK")

✅ Imports OK


## 3. Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ──────────────────────────────────────────────
#  CONFIGURATION — modifiez ces chemins
# ──────────────────────────────────────────────

# Chemin vers votre fichier SQLite
DB_PATH = "/content/drive/MyDrive/Colab Notebooks/data/DECIDON/BDProsopro/prosopography.db"   # ex: "/content/drive/MyDrive/ma_base.db"

# Dossier contenant les fichiers JSON d'occurrences (les prédictions en sortie de NER)
#JSON_INPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/data/DECIDON/JSON/"   # ex: "/content/drive/MyDrive/sessions/"
JSON_INPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT/"   # ex: "/content/drive/MyDrive/sessions/"

# Dossier de sortie pour les JSON enrichis
JSON_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT_output/"

# ──────────────────────────────────────────────
#  Schéma de la table SQLite "persons"
# ──────────────────────────────────────────────
SQL_TABLE          = "persons"
SQL_COL_ID         = "person_id"
SQL_COL_LAST_NAME  = "last_name"
SQL_COL_FIRST_NAME = "first_name"
SQL_COL_ALIAS      = "alias"
SQL_COL_BIRTH      = "birth_date"
SQL_COL_DEATH      = "death_date"
SQL_COL_WIKIDATA   = "wikidata_qid"
SQL_COL_WIKIPEDIA  = "wikipedia_url"
SQL_COL_SYCOMORE   = "sycomore_id"
SQL_COL_SENAT      = "senat_id"

# Séparateur utilisé dans la colonne 'alias' s'il y en a plusieurs.
# Mettez None si la colonne ne contient qu'un seul alias (jamais de séparateur).
ALIAS_SEPARATOR = "|"   # ex: "|", ",", ";" — à adapter selon vos données


# ──────────────────────────────────────────────
#  Filtre temporel pour la construction de la KB
# ──────────────────────────────────────────────

# Marge en années autour de SESSION_YEAR : une personne est incluse si elle
# est potentiellement vivante OU si elle a exercé un mandat dont la période
# chevauche [SESSION_YEAR - YEAR_MARGIN, SESSION_YEAR + YEAR_MARGIN]
YEAR_MARGIN = 3

# ──────────────────────────────────────────────
#  Schéma de la table SQLite "mandate"
#  → À adapter selon votre schéma réel
# ──────────────────────────────────────────────
SQL_MANDATE_TABLE        = "mandates"
SQL_MANDATE_COL_PERSON   = "person_id"
SQL_MANDATE_COL_ID       = "id"
SQL_MANDATE_COL_POSITION = "position"    # ex: "député de la Seine"
SQL_MANDATE_COL_ROLES    = "role"       # ex: "président de commission"
SQL_MANDATE_COL_START    = "start_date"
SQL_MANDATE_COL_END      = "end_date"

# ──────────────────────────────────────────────
#  Paramètres d'Entity Linking
# ──────────────────────────────────────────────
FUZZY_THRESHOLD  = 80    # score minimum RapidFuzz (0–100)
TOP_K_CANDIDATES = 5     # nb max de candidats à considérer

# Modèle SpaCy à utiliser
SPACY_MODEL = "fr_core_news_lg"

# Nombre de candidats à inclure dans la liste "persons" du JSON de sortie
# (1 = seulement le meilleur, N = les N premiers candidats enrichis)
NEL_TOP_PERSONS = 3

# classes NER à traiter
PER_LABELS = ["PER", "SPK"]

os.makedirs(JSON_INPUT_DIR, exist_ok=True)
os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)
print("✅ Configuration chargée")

✅ Configuration chargée


## 4. Inspection de la base SQLite

In [ ]:
import sqlite3
import pandas as pd

# Chemin vers votre fichier .db
db_file = DB_PATH

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Liste des tables
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

tables = [row[0] for row in cursor.fetchall()]

print(f"\nNombre de tables : {len(tables)}")

for table in tables:
    print("\n" + "=" * 80)
    print(f"TABLE : {table}")
    print("=" * 80)

    # Structure des colonnes
    print("\nColonnes :")
    cursor.execute(f"PRAGMA table_info('{table}')")
    cols = cursor.fetchall()

    for cid, name, col_type, notnull, default, pk in cols:
        print(
            f"  - {name:<30} "
            f"type={col_type:<15} "
            f"NOT NULL={bool(notnull)} "
            f"PK={bool(pk)}"
        )

    # Clés étrangères
    cursor.execute(f"PRAGMA foreign_key_list('{table}')")
    fks = cursor.fetchall()

    if fks:
        print("\nClés étrangères :")
        for fk in fks:
            print(
                f"  {fk[3]} -> {fk[2]}.{fk[4]}"
            )

    # Nombre de lignes
    try:
        cursor.execute(f"SELECT COUNT(*) FROM '{table}'")
        n = cursor.fetchone()[0]
        print(f"\nNombre d'enregistrements : {n}")
    except Exception as e:
        print(f"\nImpossible de compter les lignes : {e}")

    # Aperçu des données
    try:
        print("\nAperçu (5 premières lignes) :")
        df = pd.read_sql_query(
            f"SELECT * FROM '{table}' LIMIT 5",
            conn
        )

        if len(df) > 0:
            print(df.to_string(index=False))
        else:
            print("(table vide)")
    except Exception as e:
        print(f"Erreur de lecture : {e}")

conn.close()


Nombre de tables : 3

TABLE : legislatures

Colonnes :
  - legislature_id                 type=VARCHAR         NOT NULL=True PK=True
  - institution                    type=VARCHAR         NOT NULL=True PK=False
  - name                           type=VARCHAR         NOT NULL=True PK=False
  - start_date                     type=DATE            NOT NULL=True PK=False
  - end_date                       type=DATE            NOT NULL=True PK=False
  - wikidata_qid                   type=VARCHAR         NOT NULL=False PK=False
  - wikipedia_url                  type=VARCHAR         NOT NULL=False PK=False

Nombre d'enregistrements : 131

Aperçu (5 premières lignes) :
              legislature_id institution                                        name start_date   end_date wikidata_qid wikipedia_url
legislature_decidon_FKF5IMjT     chambre XIIe législature de la Troisième République 1919-12-08 1924-05-31         None          None
legislature_decidon_68dX8AyD     chambre XVIe législature d

### Fonction d'inspection d'une colonne de dates

In [ ]:
import sqlite3
import pandas as pd

# Chemin vers votre fichier .db
db_file = DB_PATH

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

def inspect_dates(conn, table, column):
    """
    Affiche quelques statistiques sur une colonne contenant des dates.
    Les dates peuvent être au format YYYY-MM-DD ou simplement YYYY.
    """
    print(f"\n--- {table}.{column} ---")

    query = f"""
    SELECT
        MIN(substr({column},1,4)) AS min_year,
        MAX(substr({column},1,4)) AS max_year,
        COUNT(*) AS total,
        SUM({column} IS NULL OR {column}='') AS missing
    FROM {table}
    """

    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))

    # Répartition par siècle
    query = f"""
    SELECT
        ((CAST(substr({column},1,4) AS INTEGER)-1)/100 +1) AS century,
        COUNT(*) AS n
    FROM {table}
    WHERE {column} IS NOT NULL
      AND {column}!=''
    GROUP BY century
    ORDER BY century
    """

    df = pd.read_sql_query(query, conn)

    if len(df):
        print("\nRépartition par siècle :")
        print(df.to_string(index=False))

inspect_dates(conn, SQL_TABLE, SQL_COL_BIRTH)
inspect_dates(conn, SQL_TABLE, SQL_COL_DEATH)
inspect_dates(conn, SQL_MANDATE_TABLE, SQL_MANDATE_COL_START)
inspect_dates(conn, SQL_MANDATE_TABLE, SQL_MANDATE_COL_END)

conn.close()


--- persons.birth_date ---
min_year max_year  total  missing
    1792     1936   5053        0

Répartition par siècle :
 century    n
      18    9
      19 4943
      20  101

--- persons.death_date ---
min_year max_year  total  missing
    1880     2008   5053       19

Répartition par siècle :
 century    n
      19  865
      20 4165
      21    4

--- mandates.start_date ---
min_year max_year  total  missing
    1875     1939  14131     2068

Répartition par siècle :
 century    n
      19 4468
      20 7595

--- mandates.end_date ---
min_year max_year  total  missing
    1879     1945  14131     2068

Répartition par siècle :
 century    n
      19 3585
      20 8478


### Fonction pour les mandats

In [ ]:
import sqlite3
import pandas as pd

# Chemin vers votre fichier .db
db_file = DB_PATH

conn = sqlite3.connect(db_file)
cursor = conn.cursor()


def inspect_mandats(conn, table="mandats",
                    debut="date_debut",
                    fin="date_fin"):
    print("\n" + "="*80)
    print("MANDATS")
    print("="*80)

    inspect_dates(conn, table, debut)
    inspect_dates(conn, table, fin)

    print("\nDurée des mandats")

    query = f"""
    SELECT
        COUNT(*) nb,
        AVG(CAST(substr({fin},1,4) AS INTEGER)
          - CAST(substr({debut},1,4) AS INTEGER)) AS duree_moyenne,
        MIN(CAST(substr({fin},1,4) AS INTEGER)
          - CAST(substr({debut},1,4) AS INTEGER)) AS duree_min,
        MAX(CAST(substr({fin},1,4) AS INTEGER)
          - CAST(substr({debut},1,4) AS INTEGER)) AS duree_max
    FROM {table}
    WHERE {debut} IS NOT NULL
      AND {fin} IS NOT NULL
    """

    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))

inspect_mandats(conn, SQL_MANDATE_TABLE, SQL_MANDATE_COL_START, SQL_MANDATE_COL_END)

conn.close()


MANDATS

--- mandates.start_date ---
min_year max_year  total  missing
    1875     1939  14131     2068

Répartition par siècle :
 century    n
      19 4468
      20 7595

--- mandates.end_date ---
min_year max_year  total  missing
    1879     1945  14131     2068

Répartition par siècle :
 century    n
      19 3585
      20 8478

Durée des mandats
   nb  duree_moyenne  duree_min  duree_max
12063       4.611705          0         40


### Fonction pour les personnes

In [ ]:
import sqlite3
import pandas as pd

# Chemin vers votre fichier .db
db_file = DB_PATH

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

def inspect_personnes(conn,
                      table="personnes",
                      naissance="date_naissance",
                      deces="date_deces"):

    print("\n" + "="*80)
    print("PERSONNES")
    print("="*80)

    inspect_dates(conn, table, naissance)
    inspect_dates(conn, table, deces)

    print("\nAge au décès")

    query = f"""
    SELECT
        AVG(CAST(substr({deces},1,4) AS INTEGER)
          - CAST(substr({naissance},1,4) AS INTEGER)) AS age_moyen,
        MIN(CAST(substr({deces},1,4) AS INTEGER)
          - CAST(substr({naissance},1,4) AS INTEGER)) AS age_min,
        MAX(CAST(substr({deces},1,4) AS INTEGER)
          - CAST(substr({naissance},1,4) AS INTEGER)) AS age_max
    FROM {table}
    WHERE {naissance} IS NOT NULL
      AND {deces} IS NOT NULL
    """

    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))

inspect_personnes(conn, SQL_TABLE, SQL_COL_BIRTH, SQL_COL_DEATH)

conn.close()


PERSONNES

--- persons.birth_date ---
min_year max_year  total  missing
    1792     1936   5053        0

Répartition par siècle :
 century    n
      18    9
      19 4943
      20  101

--- persons.death_date ---
min_year max_year  total  missing
    1880     2008   5053       19

Répartition par siècle :
 century    n
      19  865
      20 4165
      21    4

Age au décès
 age_moyen  age_min  age_max
 71.589591        0      151


### Répartition chronologique

In [ ]:
import sqlite3
import pandas as pd

# Chemin vers votre fichier .db
db_file = DB_PATH

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

def distribution_decennies(conn, table, column):

    query = f"""
    SELECT
        (CAST(substr({column},1,4) AS INTEGER)/10)*10 AS decennie,
        COUNT(*) AS nb
    FROM {table}
    WHERE {column} IS NOT NULL
      AND {column}!=''
    GROUP BY decennie
    ORDER BY decennie
    """

    df = pd.read_sql_query(query, conn)

    print(f"\nRépartition de {table}.{column}")
    print(df.to_string(index=False))

distribution_decennies(conn, SQL_TABLE, SQL_COL_BIRTH)
distribution_decennies(conn, SQL_TABLE, SQL_COL_DEATH)
distribution_decennies(conn, SQL_MANDATE_TABLE, SQL_MANDATE_COL_START)
distribution_decennies(conn, SQL_MANDATE_TABLE, SQL_MANDATE_COL_END)

conn.close()


Répartition de persons.birth_date
 decennie  nb
     1790   5
     1800  70
     1810 235
     1820 440
     1830 541
     1840 702
     1850 704
     1860 693
     1870 742
     1880 518
     1890 285
     1900 116
     1910   1
     1930   1

Répartition de persons.death_date
 decennie  nb
     1880 303
     1890 500
     1900 539
     1910 688
     1920 592
     1930 685
     1940 634
     1950 483
     1960 314
     1970 178
     1980  79
     1990  34
     2000   5

Répartition de mandates.start_date
 decennie   nb
     1870  286
     1880 2251
     1890 1790
     1900 1900
     1910 2032
     1920 1931
     1930 1873

Répartition de mandates.end_date
 decennie   nb
     1870    1
     1880 1672
     1890 1779
     1900 1879
     1910 2079
     1920 1851
     1930 1872
     1940  930


## 5. Chargement de la base de connaissances depuis SQLite

In [ ]:
@dataclass
class PersonEntity:
    """Représente une entité personne issue de la table SQLite 'persons'."""
    entity_id: str
    last_name: str
    first_name: str = ""
    aliases: list[str] = field(default_factory=list)
    birth_date: Optional[str] = None
    death_date: Optional[str] = None
    wikidata_qid: Optional[str] = None
    wikipedia_url: Optional[str] = None
    sycomore_id: Optional[str] = None
    senat_id: Optional[str] = None

    @property
    def name(self) -> str:
        """Nom canonique affiché : 'Prénom Nom' ou 'Nom' si pas de prénom."""
        if self.first_name:
            return f"{self.first_name} {self.last_name}".strip()
        return self.last_name

    @property
    def all_names(self) -> list[str]:
        """
        Toutes les chaînes pouvant désigner cette personne :
        - 'Prénom Nom'
        - 'Nom' seul (très fréquent dans les débats parlementaires)
        - 'P. Nom' (prénom abrégé)
        - tous les alias déclarés
        """
        names = set()
        last  = (self.last_name or "").strip()
        first = (self.first_name or "").strip()

        if last:
            names.add(last)
        if first and last:
            names.add(f"{first} {last}")
            names.add(f"{first[0]}. {last}")

        for alias in self.aliases:
            alias = alias.strip()
            if alias:
                names.add(alias)

        return [n for n in names if n]

    @property
    def description(self) -> str:
        """Courte description lisible (dates + identifiants externes)."""
        parts = []
        if self.birth_date or self.death_date:
            parts.append(f"{self.birth_date or '?'} – {self.death_date or '?'}")
        if self.wikidata_qid:
            parts.append(f"Wikidata: {self.wikidata_qid}")
        if self.sycomore_id:
            parts.append(f"Sycomore: {self.sycomore_id}")
        if self.senat_id:
            parts.append(f"Sénat: {self.senat_id}")
        return " | ".join(parts)


def _split_aliases(raw: Optional[str]) -> list[str]:
    if not raw:
        return []
    if ALIAS_SEPARATOR is None:
        return [raw.strip()] if raw.strip() else []
    return [a.strip() for a in raw.split(ALIAS_SEPARATOR) if a.strip()]


def load_knowledge_base(
    db_path: str,
    year_min: Optional[int] = None,
    year_max: Optional[int] = None,
) -> list[PersonEntity]:
    """
    Charge les entités personnes depuis la table SQLite 'persons'.

    Si year_min/year_max sont fournis, ne charge que les personnes :
      - potentiellement vivantes pendant [year_min, year_max] :
        (naissance <= year_max) ET (décès inconnu OU décès >= year_min) ;
      - OU ayant exercé un mandat (table SQL_MANDATE_TABLE) dont la période
        chevauche [year_min, year_max].

    Les dates NULL sont traitées de façon permissive : une personne sans
    date de décès connue n'est jamais exclue par ce critère.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Base SQLite introuvable : {db_path}")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (SQL_TABLE,)
    )
    if not cur.fetchone():
        conn.close()
        raise ValueError(f"Table '{SQL_TABLE}' absente de la base. Vérifiez SQL_TABLE.")

    base_select = f"""
        SELECT {SQL_COL_ID}         AS person_id,
               {SQL_COL_LAST_NAME}  AS last_name,
               {SQL_COL_FIRST_NAME} AS first_name,
               {SQL_COL_ALIAS}      AS alias,
               {SQL_COL_BIRTH}      AS birth_date,
               {SQL_COL_DEATH}      AS death_date,
               {SQL_COL_WIKIDATA}   AS wikidata_qid,
               {SQL_COL_WIKIPEDIA}  AS wikipedia_url,
               {SQL_COL_SYCOMORE}   AS sycomore_id,
               {SQL_COL_SENAT}      AS senat_id
        FROM {SQL_TABLE}
    """

    params: list = []

    if year_min is None or year_max is None:
        # Pas de filtre temporel : comportement précédent (toute la base)
        query = base_select
    else:
        # La table des mandats existe-t-elle ?
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
            (SQL_MANDATE_TABLE,)
        )
        has_mandate_table = cur.fetchone() is not None

        # Condition 1 : "vivant pendant [year_min, year_max]"
        # NB: strftime('%Y', ...) suppose des dates au format ISO 'YYYY-MM-DD'.
        # Si vos dates sont stockées comme de simples entiers (années),
        # remplacez CAST(strftime('%Y', col) AS INTEGER) par CAST(col AS INTEGER).
        alive_condition = f"""
            (
                ({SQL_COL_BIRTH} IS NULL OR CAST(strftime('%Y', {SQL_COL_BIRTH}) AS INTEGER) <= ?)
                AND
                ({SQL_COL_DEATH} IS NULL OR CAST(strftime('%Y', {SQL_COL_DEATH}) AS INTEGER) >= ?)
            )
        """
        params += [year_max, year_min]

        if has_mandate_table:
            # Condition 2 : "a un mandat chevauchant [year_min, year_max]"
            mandate_condition = f"""
                OR {SQL_COL_ID} IN (
                    SELECT {SQL_MANDATE_COL_PERSON}
                    FROM {SQL_MANDATE_TABLE}
                    WHERE (
                        {SQL_MANDATE_COL_START} IS NULL
                        OR CAST(strftime('%Y', {SQL_MANDATE_COL_START}) AS INTEGER) <= ?
                    )
                    AND (
                        {SQL_MANDATE_COL_END} IS NULL
                        OR CAST(strftime('%Y', {SQL_MANDATE_COL_END}) AS INTEGER) >= ?
                    )
                )
            """
            params += [year_max, year_min]
        else:
            mandate_condition = ""
            print(f"⚠️  Table '{SQL_MANDATE_TABLE}' introuvable : filtre basé "
                  f"uniquement sur les dates de naissance/décès.")

        query = f"{base_select} WHERE {alive_condition} {mandate_condition}"

    cur.execute(query, params)
    rows = cur.fetchall()
    conn.close()

    entities = []
    for row in rows:
        entities.append(PersonEntity(
            entity_id     = row["person_id"],
            last_name     = row["last_name"] or "",
            first_name    = row["first_name"] or "",
            aliases       = _split_aliases(row["alias"]),
            birth_date    = row["birth_date"],
            death_date    = row["death_date"],
            wikidata_qid  = row["wikidata_qid"],
            wikipedia_url = row["wikipedia_url"],
            sycomore_id   = row["sycomore_id"],
            senat_id      = row["senat_id"],
        ))

    return entities

def detect_year_range(json_dir: str, margin: int = YEAR_MARGIN):
    """
    Parcourt tous les fichiers JSON du dossier d'entrée et extrait
    l'année min et max trouvées dans le champ 'date' de chaque mention.
    Retourne (year_min, year_max) avec la marge appliquée,
    ou (None, None) si aucune date n'est trouvée.
    """
    years = []
    for json_path in glob.glob(os.path.join(json_dir, "*.json")):
        with open(json_path, encoding="utf-8") as f:
            mentions = json.load(f)
        if isinstance(mentions, dict):
            mentions = [mentions]
        for m in mentions:
            raw_date = m.get("date", "")
            if raw_date:
                try:
                    years.append(int(raw_date[:4]))
                except ValueError:
                    pass

    if not years:
        print("⚠️  Aucune date trouvée dans les fichiers JSON — filtre temporel désactivé.")
        return None, None

    year_min = min(years) - margin
    year_max = max(years) + margin
    print(f"📅 Plage temporelle détectée : {min(years)}–{max(years)} "
          f"(avec marge ±{margin} → filtre {year_min}–{year_max})")
    return year_min, year_max

# ── Appel avec filtre temporel ──────────────────────────────
YEAR_MIN, YEAR_MAX = detect_year_range(JSON_INPUT_DIR)
kb_entities = load_knowledge_base(DB_PATH, year_min=YEAR_MIN, year_max=YEAR_MAX)
print(f"✅ {len(kb_entities)} entités chargées depuis la table '{SQL_TABLE}' "
      f"(fenêtre {YEAR_MIN}–{YEAR_MAX})")
print("\nExemples :")
for e in kb_entities[:5]:
    print(f"  [{e.entity_id}] {e.name}  | variantes : {e.all_names}")

📅 Plage temporelle détectée : 1889–1936 (avec marge ±3 → filtre 1886–1939)
✅ 4911 entités chargées depuis la table 'persons' (fenêtre 1886–1939)

Exemples :
  [person_decidon_1rmQnUO6] Antoine Blanc  | variantes : ['Antoine Blanc', 'Blanc', 'A. Blanc']
  [person_decidon_ChpCy9Sa] Alphonse, Joseph Dupont  | variantes : ['Dupont', 'A. Dupont', 'Alphonse, Joseph Dupont']
  [person_decidon_v1wbIf3D] Prosper Blanc  | variantes : ['P. Blanc', 'Blanc', 'Prosper Blanc']
  [person_decidon_H1HShVlX] Henri, Antoine, Marie Germain  | variantes : ['Henri, Antoine, Marie Germain', 'Germain', 'H. Germain']
  [person_decidon_2kn3Mvy4] René Nicod  | variantes : ['Nicod', 'René Nicod', 'R. Nicod']


Ce filtre temporelle est appliqué une fois pour construire la KB utilisée sur l'ensemble du corpus JSON. Si les sessions couvrent des années très différentes (par exemple 1880 à 1920), une fenêtre unique centrée sur 1903 exclurait à tort des personnes pertinentes pour les sessions plus tardives. Dans ce cas, le plus simple est de calculer YEAR_MIN/YEAR_MAX à partir du session_date minimum et maximum trouvés dans JSON_INPUT_DIR (plus la marge), plutôt que de fixer SESSION_YEAR en dur

## 6. Construction de la Knowledge Base SpaCy

In [ ]:
# Chargement du modèle SpaCy
nlp = spacy.load(SPACY_MODEL)
print(f"✅ Modèle SpaCy chargé : {SPACY_MODEL}")

# Taille du vecteur d'entité (doit correspondre au modèle)
ENTITY_VECTOR_SIZE = nlp.vocab.vectors_length if nlp.vocab.vectors_length > 0 else 300
print(f"   Dimension des vecteurs : {ENTITY_VECTOR_SIZE}")

✅ Modèle SpaCy chargé : fr_core_news_lg
   Dimension des vecteurs : 300


In [ ]:
def build_spacy_kb(
    nlp_model: Language,
    entities: list[PersonEntity],
    vector_size: int
) -> KnowledgeBase:
    """
    Construit une KnowledgeBase SpaCy à partir des entités SQLite.
    Pour chaque entité :
      - entity_vector = vecteur SpaCy du nom canonique ('Prénom Nom')
      - alias = toutes les variantes de nom (cf. PersonEntity.all_names)
    """
    print("Construction de la KB SpaCy...")
    kb = InMemoryLookupKB(
        vocab=nlp_model.vocab,
        entity_vector_length=vector_size
    )

    print("Ajout des entités dans la KB SpaCy...")
    n_skipped = 0
    for entity in tqdm(entities):
        if not entity.name.strip():
            n_skipped += 1
            continue

        doc = nlp_model(entity.name)
        vec = doc.vector if doc.has_vector else np.zeros(vector_size, dtype="f")

        kb.add_entity(
            entity        = entity.entity_id,
            freq          = 1.0,
            entity_vector = vec,
        )

        for alias_name in entity.all_names:
            # add_alias lève une erreur si l'alias existe déjà pour la même KB ;
            # on l'ignore proprement si déjà présent (alias partagés entre entités).
            try:
                kb.add_alias(
                    alias         = alias_name,
                    entities      = [entity.entity_id],
                    probabilities = [1.0],
                )
            except ValueError:
                pass

    if n_skipped:
        print(f"⚠️  {n_skipped} entité(s) ignorée(s) (nom vide)")
    print(f"✅ KB construite : {kb.get_size_entities()} entités, {kb.get_size_aliases()} alias")
    return kb


spacy_kb = build_spacy_kb(nlp, kb_entities, ENTITY_VECTOR_SIZE)

Construction de la KB SpaCy...
Ajout des entités dans la KB SpaCy...


  0%|          | 0/4911 [00:00<?, ?it/s]/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'Blanc' already exists in the Knowledge Base.
  kb.add_alias(
  1%|          | 39/4911 [00:00<02:07, 38.31it/s]/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'Goujon' already exists in the Knowledge Base.
  kb.add_alias(
  1%|          | 49/4911 [00:01<01:58, 40.99it/s]/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'Desjardins' already exists in the Knowledge Base.
  kb.add_alias(
  1%|▏         | 68/4911 [00:01<01:41, 47.83it/s]/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'Deguise' already exists in the Knowledge Base.
  kb.add_alias(
  2%|▏         | 74/4911 [00:01<01:36, 50.16it/s]/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'F. Hugues' already exists in the Knowledge Base.
  kb.add_alias(
/tmp/ipykernel_83684/3361023303.py:38: UserWarning: [W017] Alias 'Hugues' already exists in the Knowledge Base.
  kb.

✅ KB construite : 4911 entités, 13692 alias


## 7. Moteur d'Entity Linking (fuzzy + vecteurs SpaCy)

La stratégie combine :
1. **Matching exact** sur un nom/alias enregistré (`all_names`)
2. **Matching flou** (RapidFuzz) sur tous les noms/alias de la KB
3. **Score vectoriel** SpaCy pour départager les candidats proches

In [ ]:
# Index plat : nom_normalisé (minuscule) → entity_id
def build_name_index(entities: list[PersonEntity]) -> dict[str, str]:
    index = {}
    for e in entities:
        for name in e.all_names:
            index[name.lower().strip()] = e.entity_id
    return index


name_index       = build_name_index(kb_entities)
all_indexed_names = list(name_index.keys())
entity_map: dict[str, PersonEntity] = {e.entity_id: e for e in kb_entities}


def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))


def link_entity(
    str_ref: str,
    nlp_model: Language,
    n_index: dict[str, str],
    indexed_names: list[str],
    e_map: dict[str, PersonEntity],
    fuzzy_threshold: int = FUZZY_THRESHOLD,
    top_k: int = TOP_K_CANDIDATES,
) -> dict:
    """
    Tente de lier str_ref à une entité de la KB.
    Retourne un dict avec :
      - entity_id, person  (objet PersonEntity ou None)
      - method   : 'exact' | 'fuzzy' | 'vector' | 'nil'
      - score    : score de confiance (0.0–1.0)
      - candidates : liste des candidats considérés
    """
    query = str_ref.strip()
    query_lower = query.lower()

    # ── 1. Matching exact ──────────────────────────────────────────────────
    if query_lower in n_index:
        eid = n_index[query_lower]
        return {
            "entity_id": eid,
            "person"   : e_map[eid],
            "method"   : "exact",
            "score"    : 1.0,
            "candidates": [{"id": eid, "name": e_map[eid].name, "score": 1.0}],
        }

    # ── 2. Matching flou (RapidFuzz) ───────────────────────────────────────
    fuzzy_results = process.extract(
        query_lower,
        indexed_names,
        scorer       = fuzz.token_sort_ratio,
        limit        = top_k,
        score_cutoff = fuzzy_threshold,
    )

    if not fuzzy_results:
        return {
            "entity_id": None,
            "person"   : None,
            "method"   : "nil",
            "score"    : 0.0,
            "candidates": [],
        }

    # Dédoublonnage des candidats (plusieurs alias → même entité)
    seen_ids = {}
    for name_match, score, _ in fuzzy_results:
        eid = n_index[name_match]
        if eid not in seen_ids or score > seen_ids[eid]["score"]:
            seen_ids[eid] = {
                "id"         : eid,
                "name"       : e_map[eid].name,
                "alias_match": name_match,
                "score"      : score / 100.0,
            }
    candidates = sorted(seen_ids.values(), key=lambda x: x["score"], reverse=True)

    best = candidates[0]
    if len(candidates) == 1 or best["score"] >= 0.95:
        eid = best["id"]
        return {
            "entity_id": eid,
            "person"   : e_map[eid],
            "method"   : "fuzzy",
            "score"    : best["score"],
            "candidates": candidates,
        }

    # ── 3. Désambiguïsation vectorielle ───────────────────────────────────
    query_vec = nlp_model(query).vector
    best_combined = -1.0
    best_id       = candidates[0]["id"]

    for cand in candidates:
        cand_vec = nlp_model(e_map[cand["id"]].name).vector
        cos = cosine_similarity(query_vec, cand_vec)
        combined = 0.6 * cand["score"] + 0.4 * ((cos + 1) / 2)
        cand["combined_score"] = combined
        if combined > best_combined:
            best_combined = combined
            best_id        = cand["id"]

    eid = best_id
    return {
        "entity_id": eid,
        "person"   : e_map[eid],
        "method"   : "vector",
        "score"    : best_combined,
        "candidates": candidates,
    }


print("✅ Moteur d'entity linking prêt")

✅ Moteur d'entity linking prêt


## 8. Traitement des fichiers JSON


In [ ]:
def _build_person_entry(rank: int, person: PersonEntity, score: float,
                        method: str) -> dict:
    return {
        "rank"          : rank,
        "match_strategy": method,
        "link_method"   : method,
        "link_score"    : round(score, 4),
        "person_id"     : person.entity_id,
        "wikidata_url"  : (f"https://www.wikidata.org/wiki/{person.wikidata_qid}"
                           if person.wikidata_qid else None),
        "first_name"    : person.first_name,
        "last_name"     : person.last_name,
        "alias"         : person.aliases[0] if person.aliases else "",
    }

def process_nel_json(json_path: str, output_dir: str) -> dict:
    """
    Traite un fichier JSON au format NEL (liste de mentions annotées).
    Pour chaque mention de classe PER :
      - applique le linking
      - AJOUTE le(s) candidat(s) trouvés à la liste 'persons' existante
        (sans écraser les entrées déjà présentes)
      - laisse 'id_entity' et 'wikidata_url' intacts (valeurs d'origine)
    Écrit le JSON enrichi dans output_dir et retourne les stats.
    """
    with open(json_path, encoding="utf-8") as f:
        mentions = json.load(f)

    if isinstance(mentions, dict):
        mentions = [mentions]

    stats = {"total": 0, "exact": 0, "fuzzy": 0, "vector": 0, "nil": 0}
    enriched_mentions = []

    for mention in mentions:
        # On ne traite que les mentions de personnes
        if mention.get("classe") not in PER_LABELS:
            enriched_mentions.append(mention)
            continue

        str_ref = mention.get("entity", "").strip()
        stats["total"] += 1

        # ── Linking ────────────────────────────────────────────────────────
        result = link_entity(
            str_ref, nlp, name_index, all_indexed_names, entity_map,
            top_k=NEL_TOP_PERSONS,
        )
        stats[result["method"]] = stats.get(result["method"], 0) + 1

        # ── Construction des nouveaux candidats à ajouter ──────────────────
        new_persons = []
        for rank_offset, cand in enumerate(result["candidates"][:NEL_TOP_PERSONS], start=1):
            cand_person = entity_map.get(cand["id"])
            if cand_person is None:
                continue
            cand_score = cand.get("combined_score", cand["score"])
            new_persons.append(
                _build_person_entry(rank_offset, cand_person, cand_score, result["method"])
            )

        # ── Fusion avec la liste 'persons' existante ───────────────────────
        # On repart de l'élément d'entrée (copie) pour ne rien écraser
        enriched = dict(mention)

        existing_persons = list(mention.get("persons") or [])

        # Calcul du rang de départ pour les nouveaux candidats :
        # ils s'ajoutent après les entrées déjà présentes
        next_rank = (max((p.get("rank", 0) for p in existing_persons), default=0) + 1)
        for i, p in enumerate(new_persons):
            p["rank"] = next_rank + i

        enriched["persons"] = existing_persons + new_persons

        # 'id_entity' et 'wikidata_url' : on conserve strictement les valeurs
        # d'origine (null ou déjà renseignées par un processus NEL précédent)
        enriched["id_entity"]    = mention.get("id_entity")
        enriched["wikidata_url"] = mention.get("wikidata_url")

        enriched_mentions.append(enriched)

    # ── Écriture du fichier de sortie ──────────────────────────────────────
    out_filename = Path(json_path).stem + "_linked.json"
    out_path     = os.path.join(output_dir, out_filename)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(enriched_mentions, f, ensure_ascii=False, indent=2)

    return stats

print("✅ Fonction de traitement prête")

✅ Fonction de traitement prête


In [ ]:
json_files = glob.glob(os.path.join(JSON_INPUT_DIR, "*.json"))

if not json_files:
    print(f"⚠️  Aucun fichier JSON trouvé dans : {JSON_INPUT_DIR}")
    print("    → Uploadez vos fichiers via la section 4, ou ajustez JSON_INPUT_DIR.")
else:
    print(f"📂 {len(json_files)} fichier(s) à traiter\n")
    global_stats = {"total": 0, "exact": 0, "fuzzy": 0, "vector": 0, "nil": 0}

    for json_file in tqdm(json_files, desc="Fichiers traités"):
        stats = process_nel_json(json_file, JSON_OUTPUT_DIR)
        for k in global_stats:
            global_stats[k] += stats.get(k, 0)

    print("\n── Résumé global ─────────────────────────────")
    print(f"  Total mentions PER     : {global_stats['total']}")
    resolved = global_stats['total'] - global_stats['nil']
    pct = (resolved / global_stats['total'] * 100) if global_stats['total'] else 0
    print(f"  Résolues               : {resolved} ({pct:.1f}%)")
    print(f"    via exact match      : {global_stats['exact']}")
    print(f"    via fuzzy match      : {global_stats['fuzzy']}")
    print(f"    via vecteurs         : {global_stats['vector']}")
    print(f"  Non résolues (NIL)     : {global_stats['nil']}")
    print(f"\n✅ Fichiers enrichis écrits dans : {JSON_OUTPUT_DIR}")



📂 2 fichier(s) à traiter



Fichiers traités: 100%|██████████| 2/2 [00:16<00:00,  8.14s/it]


── Résumé global ─────────────────────────────
  Total mentions PER     : 4686
  Résolues               : 2862 (61.1%)
    via exact match      : 2257
    via fuzzy match      : 534
    via vecteurs         : 71
  Non résolues (NIL)     : 1824

✅ Fichiers enrichis écrits dans : /content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT_output/


## 9. Inspection des résultats

In [ ]:
output_files = glob.glob(os.path.join(JSON_OUTPUT_DIR, "*_linked.json"))

if not output_files:
    print("Aucun fichier de sortie trouvé — exécutez d'abord la section 9.")
else:
    with open(output_files[0], encoding="utf-8") as f:
        mentions = json.load(f)

    per_mentions = [m for m in mentions if m.get("classe") in PER_LABELS]
    print(f"Fichier : {output_files[0]}")
    print(f"{len(per_mentions)} mention(s) PER/SPK\n")
    print(f"{'entity':<25} {'name résolue':<28} {'méthode':<8} {'score':>6}  {'person_id':<28} wikidata_url")
    print("─" * 120)
    for m in per_mentions:
        persons = m.get("persons", [])
        best    = persons[0] if persons else {}
        name    = f"{best.get('first_name','')} {best.get('last_name','')}".strip() or "— NIL —"
        pid     = best.get("person_id") or ""
        wiki    = m.get("wikidata_url") or ""
        method  = m.get("link_method", "nil")
        score   = m.get("link_score", 0.0)
        print(
            f"{m.get('entity',''):<25} {name:<28} {method:<8} {score:>6.3f}  {pid:<28} {wiki}"
        )

Fichier : /content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT_output/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked.json
4686 mention(s) PER/SPK

entity                    name résolue                 méthode   score  person_id                    wikidata_url
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Henri Brisson             Henri Brisson                nil       0.000  person_decidon_95vFBkGw      
le ministre des affaires étrangères Alexandre Ribot              nil       0.000  person_decidon_Wzp1SOFB      
Henri Brisson             Henri Brisson                nil       0.000  person_decidon_95vFBkGw      
le ministre des affaires étrangères Alexandre Ribot              nil       0.000  person_decidon_Wzp1SOFB      
du Breuil de Saint-Germain Albert Moreau Du Breuil de Saint-Germain nil       0.000  person_decidon_6tw4ByXz      
du Breuil de Saint-Germain Albert Moreau Du Breuil de Saint-Germain nil 

## 10. Test rapide sur une occurrence unique

In [ ]:
# ── Test interactif ──────────────────────────────────────
test_str_ref  = "Henri Brisson"   # ← modifiez ici
test_year     = 1890              # ← année de la séance (pour info, non utilisée dans le matching)

result = link_entity(test_str_ref, nlp, name_index, all_indexed_names, entity_map)
person = result["person"]

print(f"Requête      : '{test_str_ref}'")
print(f"Méthode      : {result['method']}  (score={result['score']:.4f})")
print()
if person:
    print(f"✅ Entité résolue")
    print(f"   person_id   : {person.entity_id}")
    print(f"   Nom         : {person.name}")
    print(f"   Dates       : {person.birth_date or '?'} – {person.death_date or '?'}")
    print(f"   Wikidata    : {person.wikidata_qid}")
    print(f"   Sycomore    : {person.sycomore_id}")
    print(f"   Sénat       : {person.senat_id}")
else:
    print("❌ Non résolu (NIL)")

if result["candidates"]:
    print(f"\nTop candidats :")
    for c in result["candidates"]:
        cand_person = entity_map.get(c["id"])
        dates = ""
        if cand_person:
            dates = f"{cand_person.birth_date or '?'} – {cand_person.death_date or '?'}"
        print(f"  [{c['score']:.4f}] {c['name']:<28} {dates}  ({c['id']})")

Requête      : 'Henri Brisson'
Méthode      : exact  (score=1.0000)

✅ Entité résolue
   person_id   : person_decidon_95vFBkGw
   Nom         : Henri Brisson
   Dates       : 1835-07-31 – 1912-04-13
   Wikidata    : Q669725
   Sycomore    : 1240.0
   Sénat       : None

Top candidats :
  [1.0000] Henri Brisson                1835-07-31 – 1912-04-13  (person_decidon_95vFBkGw)


## 11. Consolidation : vote et remplissage de id_entity / wikidata_url

In [ ]:
def consolidate_mentions(linked_json_path: str, output_dir: str) -> dict:
    """
    Lit un fichier JSON de sortie NEL et consolide chaque mention PER :

    - vote = 2 : exactement 2 candidats dans 'persons' avec le même person_id
    - vote = 1 : exactement 1 candidat dans 'persons' (quel que soit son rang)
    - vote = None : tous les autres cas (0 candidats, ou 2+ candidats différents)

    Quand vote vaut 1 ou 2, remplit 'id_entity' et 'wikidata_url'
    à partir du candidat retenu. Sinon, laisse les valeurs d'origine.

    Écrit le fichier consolidé avec le suffixe _consolidated.json.
    """
    with open(linked_json_path, encoding="utf-8") as f:
        mentions = json.load(f)

    stats = {"vote_2": 0, "vote_1": 0, "vote_null": 0, "non_per": 0}
    consolidated = []

    for mention in mentions:
        if mention.get("classe") not in PER_LABELS:
            stats["non_per"] += 1
            consolidated.append(mention)
            continue

        enriched = dict(mention)
        persons  = mention.get("persons") or []

        # On ne considère que les entrées qui ont un person_id renseigné
        valid_persons = [p for p in persons if p.get("person_id")]

        # ── Calcul du vote ─────────────────────────────────────────────────
        if len(valid_persons) == 0:
            # Aucun candidat : pas de résolution possible
            vote = None

        elif len(valid_persons) == 1:
            # Un seul candidat toutes sources confondues
            vote = 1

        elif len(valid_persons) == 2:
            # Exactement 2 candidats : vote = 2 seulement s'ils pointent
            # vers la même personne, sinon ambiguïté → null
            ids = [p["person_id"] for p in valid_persons]
            vote = 2 if ids[0] == ids[1] else None

        else:
            # Plus de 2 candidats : on vérifie si tous partagent le même person_id
            ids = [p["person_id"] for p in valid_persons]
            if len(set(ids)) == 1:
                # Tous concordent (ex: 3 sources, même personne)
                vote = len(ids)
            else:
                vote = None

        enriched["vote"] = vote

        # ── Remplissage de id_entity et wikidata_url ───────────────────────
        if vote is not None:
            # On prend le premier candidat valide comme référence
            ref = valid_persons[0]
            enriched["id_entity"]    = ref.get("person_id")
            enriched["wikidata_url"] = ref.get("wikidata_url")
        else:
            # Ambiguïté ou absence : on conserve les valeurs d'origine
            enriched["id_entity"]    = mention.get("id_entity")
            enriched["wikidata_url"] = mention.get("wikidata_url")

        # Accumulation des stats
        if vote == 2:
            stats["vote_2"] += 1
        elif vote == 1:
            stats["vote_1"] += 1
        else:
            stats["vote_null"] += 1

        consolidated.append(enriched)

    # ── Écriture du fichier consolidé ──────────────────────────────────────
    out_filename = Path(linked_json_path).stem + "_consolidated.json"
    out_path     = os.path.join(output_dir, out_filename)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(consolidated, f, ensure_ascii=False, indent=2)

    return stats


# ── Traitement de tous les fichiers *_linked.json ──────────────────────────
linked_files = glob.glob(os.path.join(JSON_OUTPUT_DIR, "*_linked.json"))

if not linked_files:
    print("⚠️  Aucun fichier *_linked.json trouvé — exécutez d'abord la section 9.")
else:
    print(f"📂 {len(linked_files)} fichier(s) à consolider\n")
    total_stats = {"vote_2": 0, "vote_1": 0, "vote_null": 0, "non_per": 0}

    for linked_file in tqdm(linked_files, desc="Consolidation"):
        stats = consolidate_mentions(linked_file, JSON_OUTPUT_DIR)
        for k in total_stats:
            total_stats[k] += stats.get(k, 0)

    total_per = total_stats["vote_2"] + total_stats["vote_1"] + total_stats["vote_null"]
    resolved  = total_stats["vote_2"] + total_stats["vote_1"]
    pct       = (resolved / total_per * 100) if total_per else 0

    print("\n── Résumé consolidation ──────────────────────────────")
    print(f"  Mentions PER traitées  : {total_per}")
    print(f"  Résolues               : {resolved} ({pct:.1f}%)")
    print(f"    vote = 2 (consensus) : {total_stats['vote_2']}")
    print(f"    vote = 1 (unique)    : {total_stats['vote_1']}")
    print(f"  Non résolues (vote=∅)  : {total_stats['vote_null']}")
    print(f"\n✅ Fichiers consolidés écrits dans : {JSON_OUTPUT_DIR}")

📂 2 fichier(s) à consolider



Consolidation: 100%|██████████| 2/2 [00:01<00:00,  1.88it/s]


── Résumé consolidation ──────────────────────────────
  Mentions PER traitées  : 4686
  Résolues               : 3075 (65.6%)
    vote = 2 (consensus) : 2060
    vote = 1 (unique)    : 1015
  Non résolues (vote=∅)  : 1611

✅ Fichiers consolidés écrits dans : /content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT_output/


## 12. Export CSV consolidé

In [ ]:
def consolidated_to_csv(consolidated_dir: str, csv_path: str) -> int:
    """
    Produit un CSV à partir de tous les fichiers *_consolidated.json.
    Une ligne par mention PER.
    - Les candidats sont aplatis en colonnes persons_1_*, persons_2_*
      selon le nombre de candidats effectivement présents dans le fichier.
    - 'vote_result' synthétise la décision finale :
        'resolved_consensus'  → vote=2 (deux candidats identiques)
        'resolved_unique'     → vote=1 (un seul candidat)
        'ambiguous'           → vote=None, plusieurs candidats différents
        'unresolved'          → aucun candidat
    Retourne le nombre total de lignes écrites.
    """
    consolidated_files = glob.glob(
        os.path.join(consolidated_dir, "*_consolidated.json")
    )
    if not consolidated_files:
        print("⚠️  Aucun fichier *_consolidated.json trouvé.")
        return 0

    def extract_person_fields(p: dict, prefix: str) -> dict:
        mandates = p.get("mandates") or []
        position = mandates[0].get("position", "") if mandates else ""
        roles = mandates[0].get("roles", "") if mandates else ""
        return {
            f"{prefix}rank"           : p.get("rank", ""),
            f"{prefix}score"          : p.get("score", ""),
            f"{prefix}match_strategy" : p.get("match_strategy", ""),
            f"{prefix}link_method"    : p.get("link_method", ""),
            f"{prefix}link_score"     : p.get("link_score", ""),
            f"{prefix}person_id"      : p.get("person_id", ""),
            f"{prefix}wikidata_url"   : p.get("wikidata_url", ""),
            f"{prefix}first_name"     : p.get("first_name", ""),
            f"{prefix}last_name"      : p.get("last_name", ""),
            f"{prefix}position"       : position,
            f"{prefix}roles"       : roles,
        }

    def vote_result_label(vote, n_valid: int) -> str:
        if n_valid == 0:
            return "unresolved"
        if vote == 2:
            return "resolved_consensus"
        if vote == 1:
            return "resolved_unique"
        return "ambiguous"

    # ── Collecte de toutes les lignes en mémoire ───────────────────────────
    # Nécessaire pour déterminer le nombre max de candidats
    # avant d'écrire l'en-tête CSV.
    all_rows = []
    max_persons = 0

    for consolidated_file in sorted(consolidated_files):
        with open(consolidated_file, encoding="utf-8") as f:
            mentions = json.load(f)

        for mention in mentions:
            if mention.get("classe") not in PER_LABELS:
                continue

            persons = mention.get("persons") or []
            valid   = sorted(
                [p for p in persons if p.get("person_id")],
                key=lambda p: p.get("rank", 999)
            )
            vote    = mention.get("vote")
            n_valid = len(valid)
            max_persons = max(max_persons, n_valid)

            row = {
                # ── Champs d'origine ───────────────────────────────────────
                "uuid"          : mention.get("uuid", ""),
                "classe"        : mention.get("classe", ""),
                "entity"        : mention.get("entity", ""),
                "span"          : (f"{mention['span'][0]}:{mention['span'][1]}"
                                   if mention.get("span") else ""),
                "contexte"      : mention.get("contexte", ""),
                "task_id"       : mention.get("task_id", ""),
                "annotation_id" : mention.get("annotation_id", ""),
                "seance"        : mention.get("seance", ""),
                "date"          : mention.get("date", ""),
                "source"        : mention.get("source", ""),
                "id_entity"     : mention.get("id_entity") or "",
                "wikidata_url"  : mention.get("wikidata_url") or "",
                "vote"          : vote if vote is not None else "",
                "vote_result"   : vote_result_label(vote, n_valid),
            }

            # Ajout des candidats aplatis
            for i, p in enumerate(valid, start=1):
                row.update(extract_person_fields(p, f"persons_{i}_"))

            all_rows.append(row)

    # ── Construction du schéma dynamique ──────────────────────────────────
    person_fields = [
        "rank", "score", "match_strategy", "link_method", "link_score",
        "person_id", "wikidata_url", "first_name", "last_name", "position", "roles"
    ]
    fieldnames = [
        "uuid", "classe", "entity", "span", "contexte",
        "task_id", "annotation_id", "seance", "date", "source",
        "id_entity", "wikidata_url", "vote", "vote_result",
    ]
    for i in range(1, max_persons + 1):
        for field in person_fields:
            fieldnames.append(f"persons_{i}_{field}")

    # ── Écriture du CSV ────────────────────────────────────────────────────
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames,
                                extrasaction="ignore", delimiter="\t",
                                restval="")   # remplit "" pour les colonnes absentes
        writer.writeheader()
        writer.writerows(all_rows)

    return len(all_rows)


# ── Appel ──────────────────────────────────────────────────────────────────
csv_output_path = "nel_consolidated.csv"
n = consolidated_to_csv(JSON_OUTPUT_DIR, csv_output_path)
print(f"✅ {n} lignes écrites dans : {csv_output_path}")

try:
    from google.colab import files
    files.download(csv_output_path)
except ImportError:
    print("(Pas dans Colab — fichier disponible localement)")

✅ 4686 lignes écrites dans : nel_consolidated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Enrichissement par paires PER/FCT contiguës (intra-séance)

In [ ]:
from collections import defaultdict

def extract_per_fct_pairs(json_path: str) -> list[dict]:
    """
    Extrait les paires (PER/SPK, FCT) depuis un fichier JSON Label Studio,
    uniquement via les relations explicites 'function_of' (from_id → to_id).
    """
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        data = [data]

    pairs = []

    for doc in data:
        image_url = (
            doc.get("image_url")
            or doc.get("data", {}).get("image_url", "")
        )
        chamber = (
            doc.get("chamber")
            or doc.get("data", {}).get("meta", {}).get("chamber", "")
        )
        date_iso = (
            doc.get("date_iso")
            or doc.get("data", {}).get("meta", {}).get("date_iso", "")
        )

        annotations = doc.get("annotations", [])
        if not annotations:
            continue

        result_items = annotations[0].get("result", [])

        # ── Index des spans par id ─────────────────────────────────────────
        span_by_id: dict[str, dict] = {}
        for item in result_items:
            if item.get("type") != "labels":
                continue
            val = item["value"]
            span_by_id[item["id"]] = {
                "id"    : item["id"],
                "start" : val.get("start"),
                "end"   : val.get("end"),
                "text"  : val.get("text", "").strip(),
                "labels": val.get("labels", []),
            }

        # ── Relations explicites 'function_of' uniquement ─────────────────
        relations = [
            item for item in result_items
            if item.get("type") == "relation"
            and "function_of" in item.get("labels", [])
        ]

        seen_pairs: set[tuple] = set()

        for rel in relations:
            span_from = span_by_id.get(rel.get("from_id"))
            span_to   = span_by_id.get(rel.get("to_id"))

            if not span_from or not span_to:
                continue

            from_is_per = any(l in ("PER", "SPK") for l in span_from["labels"])
            to_is_fct   = "FCT" in span_to["labels"]
            from_is_fct = "FCT" in span_from["labels"]
            to_is_per   = any(l in ("PER", "SPK") for l in span_to["labels"])

            if from_is_per and to_is_fct:
                per_span, fct_span = span_from, span_to
            elif from_is_fct and to_is_per:
                per_span, fct_span = span_to, span_from
            else:
                continue

            key = (per_span["id"], fct_span["id"])
            if key in seen_pairs:
                continue
            seen_pairs.add(key)

            pair_str = (
                f"{per_span['start']}-{per_span['end']}"
                f"-{per_span['text']}-PER"
                f" | "
                f"{fct_span['start']}-{fct_span['end']}"
                f"-{fct_span['text']}-FCT"
                f" | "
                f"{image_url}"
                f" | "
                f"{chamber}"
                f" | "
                f"{date_iso}"
            )
            pairs.append({
                "per_start": per_span["start"],
                "per_end"  : per_span["end"],
                "per_text" : per_span["text"],
                "fct_start": fct_span["start"],
                "fct_end"  : fct_span["end"],
                "fct_text" : fct_span["text"],
                "image_url": image_url,
                "chamber"  : chamber,
                "date_iso"     : date_iso,
                "pair_str" : pair_str,
            })

    return pairs


# ── Extraction depuis tous les fichiers JSON d'entrée ─────────────────────
all_pairs: list[dict] = []
json_files = glob.glob(os.path.join(JSON_INPUT_DIR, "*.json"))

for json_file in tqdm(json_files, desc="Extraction paires PER/FCT"):
    all_pairs.extend(extract_per_fct_pairs(json_file))

pairs_index: dict[str, list[dict]] = defaultdict(list)
for pair in all_pairs:
    key = pair["per_text"].lower().strip()
    pairs_index[key].append(pair)

print(f"✅ {len(all_pairs)} paire(s) PER/FCT extraites (relations function_of uniquement)")
print(f"   {len(pairs_index)} mention(s) PER/SPK uniques indexées")
print("\nExemples :")
for key, ps in list(pairs_index.items())[:83]:
    for p in ps:
        print(f"  '{p['per_text']}' ↔ '{p['fct_text']}' ({p['chamber']} {p['date_iso']} - {p['image_url']})")

Extraction paires PER/FCT: 100%|██████████| 2/2 [00:02<00:00,  1.16s/it]

✅ 188 paire(s) PER/FCT extraites (relations function_of uniquement)
   83 mention(s) PER/SPK uniques indexées

Exemples :
  'le président du conseil' ↔ 'ministre de la guerre' (Chambre des députés 1890-06-24 - https://lsd.mezanno.xyz/data/local-files/?d=DECIDON-NER-inextenso-chandra/chambre_deputes/1890/bpt6k64574218/p000.jpg)
  'le président du conseil' ↔ 'ministre de l'intérieur' (Sénat 1904-07-05 - https://lsd.mezanno.xyz/data/local-files/?d=DECIDON-NER-inextenso-chandra/senat/1904/bpt6k64837318/p000.jpg)
  'le président du conseil' ↔ 'ministre de l'intérieur et des finances' (Sénat 1904-07-05 - https://lsd.mezanno.xyz/data/local-files/?d=DECIDON-NER-inextenso-chandra/senat/1904/bpt6k64837318/p014.jpg)
  'le président du conseil' ↔ 'ministre du commerce, de l'industrie et des colonies' (Sénat 1889-05-21 - https://lsd.mezanno.xyz/data/local-files/?d=DECIDON-NER-inextenso-chandra/senat/1889/bpt6k6383360c/p000.jpg)
  'le président du conseil' ↔ 'ministre du commerce, de l'industrie et 

In [ ]:
# Noms des colonnes issues du découpage de per_fct_pair
FCT_PAIR_COLS = [
    "per_fct_per_span",      # ex: 849-855-Nogaro-PER
    "per_fct_fct_span",      # ex: 857-867-rapporteur-FCT
    "per_fct_image_url",     # ex: https://lsd.mezanno.xyz/...
    "per_fct_chamber",       # ex: Chambre des députés
    "per_fct_date",          # ex: 1929-11-28
]

def match_entity_to_fct(entity: str, fct_text: str) -> tuple[bool, str, float]: #,nlp_model: Language
    """
    Tente de rapprocher 'entity' (colonne CSV) et 'fct_text' (membre FCT
    d'une paire) via trois stratégies successives.

    Retourne (matched, method, score).
    """
    e_norm   = entity.lower().strip()
    fct_norm = fct_text.lower().strip()

    # ── 1. Match exact après normalisation ────────────────────────────────
    if e_norm == fct_norm or e_norm in fct_norm or fct_norm in e_norm:
        return True, "exact", 1.0

    # ── 2. Match flou (RapidFuzz token_sort_ratio) ────────────────────────
    #fuzzy_score = fuzz.token_sort_ratio(e_norm, fct_norm)
    #if fuzzy_score >= FCT_FUZZY_THRESHOLD:
    #    return True, "fuzzy", fuzzy_score / 100.0

    # ── 3. Match vectoriel (cosinus SpaCy) ────────────────────────────────
    #vec_entity = nlp_model(entity).vector
    #vec_fct    = nlp_model(fct_text).vector
    #cos = cosine_similarity(vec_entity, vec_fct)
    #if cos >= FCT_VECTOR_THRESHOLD:
    #    return True, "vector", cos

    return False, "none", 0.0


def normalize_chamber(value: str) -> str:
    """
    Normalise la valeur de chambre pour permettre la comparaison
    entre le CSV ('seance') et le JSON ('chamber').
    Ex: 'Sénat' → 'senat', 'Chambre des députés' → 'deputes'
    """
    value = value.lower().strip()
    # Variantes possibles selon les sources
    if "sénat" in value or "senat" in value:
        return "senat"
    if "député" in value or "depute" in value or "chambre" in value or "chambre des députés" in value:
        return "deputes"
    return value

def enrich_csv_with_pairs(
    csv_input_path: str,
    csv_output_path: str,
    pairs_idx: dict[str, list[dict]],
) -> dict:
    stats = {
        "total": 0, "matched": 0, "unmatched": 0,
        "match_exact": 0, "match_fuzzy": 0, "match_vector": 0,
    }

    with open(csv_input_path, newline="", encoding="utf-8") as f:
        reader     = csv.DictReader(f, delimiter="\t")
        fieldnames = list(reader.fieldnames or [])
        rows       = list(reader)

    # ── Insertion des nouvelles colonnes juste après 'vote_result' ─────────
    # On remplace l'ancienne colonne 'per_fct_pair' (si présente) par les
    # colonnes découpées + les colonnes de matching
    new_cols = FCT_PAIR_COLS + ["per_fct_match_method", "per_fct_match_score"]

    # Suppression de l'éventuelle colonne per_fct_pair héritée
    if "per_fct_pair" in fieldnames:
        fieldnames.remove("per_fct_pair")

    if "vote_result" in fieldnames:
        insert_at = fieldnames.index("vote_result") + 1
        for i, col in enumerate(new_cols):
            if col not in fieldnames:
                fieldnames.insert(insert_at + i, col)
    else:
        for col in new_cols:
            if col not in fieldnames:
                fieldnames.append(col)

    # ── Aplatissement de pairs_idx en liste dédupliquée ───────────────────
    all_pairs_flat: list[dict] = []
    seen = set()
    for pair_list in pairs_idx.values():
        for pair in pair_list:
            key = id(pair)
            if key not in seen:
                seen.add(key)
                all_pairs_flat.append(pair)

    def split_pair_str(pair_str: str) -> dict:
        """
        Découpe pair_str sur le séparateur ' | ' et retourne un dict
        avec les colonnes FCT_PAIR_COLS. Les segments manquants sont
        remplis avec une chaîne vide.
        """
        segments = [s.strip() for s in pair_str.split(" | ")]
        return {
            col: segments[i] if i < len(segments) else ""
            for i, col in enumerate(FCT_PAIR_COLS)
        }

    enriched_rows = []
    for row in rows:
        stats["total"] += 1

        entity     = row.get("entity", "").strip()
        row_seance = normalize_chamber(row.get("seance", ""))
        row_date   = row.get("date", "").strip()
        vote       = row.get("vote_result", "").strip()

        # Initialisation des colonnes découpées à vide
        for col in new_cols:
            row.setdefault(col, "")

        if vote in ("ambiguous", "unresolved"):
            best_pair   = None
            best_method = "none"
            best_score  = 0.0

            date_chamber_candidates = [
                p for p in all_pairs_flat
                if normalize_chamber(p.get("chamber", "")) == row_seance
                and p.get("date_iso", "").strip() == row_date
            ]

            for pair in date_chamber_candidates:
                matched, method, score = match_entity_to_fct(
                    entity, pair["fct_text"]
                )
                if matched and score > best_score:
                    best_pair   = pair
                    best_method = method
                    best_score  = score
                    if method == "exact":
                        break

            if best_pair:
                row["vote_result"]          = "intra-seance"
                row["per_fct_match_method"] = best_method
                row["per_fct_match_score"]  = round(best_score, 4)
                # Découpage du pair_str en colonnes individuelles
                row.update(split_pair_str(best_pair["pair_str"]))
                stats["matched"] += 1
                stats[f"match_{best_method}"] += 1
            else:
                stats["unmatched"] += 1

        enriched_rows.append(row)

    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames,
                                extrasaction="ignore", delimiter="\t",
                                restval="")
        writer.writeheader()
        writer.writerows(enriched_rows)

    return stats


# ── Appel ──────────────────────────────────────────────────────────────────
csv_input_path    = JSON_OUTPUT_DIR + "nel_consolidated.csv"
csv_enriched_path = JSON_OUTPUT_DIR + "nel_consolidated_intra.csv"

stats = enrich_csv_with_pairs(csv_input_path, csv_enriched_path, pairs_index)

print(f"✅ CSV enrichi écrit : {csv_enriched_path}")
print(f"\n── Résumé ────────────────────────────────────────")
print(f"  Lignes totales          : {stats['total']}")
print(f"  Matchées                : {stats['matched']}")
print(f"    via exact             : {stats['match_exact']}")
print(f"    via fuzzy             : {stats['match_fuzzy']}")
print(f"    via vecteurs          : {stats['match_vector']}")
print(f"  Non matchées            : {stats['unmatched']}")

try:
    from google.colab import files
    files.download(csv_enriched_path)
except ImportError:
    print("(Pas dans Colab — fichier disponible localement)")

✅ CSV enrichi écrit : /content/drive/MyDrive/Colab Notebooks/data/DECIDON/VT_output/nel_consolidated_intra.csv

── Résumé ────────────────────────────────────────
  Lignes totales          : 4686
  Matchées                : 130
    via exact             : 130
    via fuzzy             : 0
    via vecteurs          : 0
  Non matchées            : 1481


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>